In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math 
%matplotlib qt5

QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-amr'


In [2]:
point_array = ["1","2","3","4","5","6","7","7.1","7.2","7.3","7.4","7.5","8","8.1","8.2","8.3","10.1","10.2"]

In [3]:
def get_column_average(bags_path):
  X = []
  Y = []
  Z = []
  for point in point_array:
    csv_file_path=bags_path+f"point_{point}/hedgehog_pos/hedgehog_pos.csv"
    # # Read the CSV file (replace 'data.csv' with your file path)
    df = pd.read_csv(csv_file_path,usecols=['x_m','y_m','z_m'])
    # # Calculate column averages
    column_averages = df.mean()
    X.append(np.round(column_averages['x_m'],4))
    Y.append(np.round(column_averages['y_m'],4))
    Z.append(np.round(column_averages['z_m'],4))
  return [X,Y,Z]

In [ ]:


# points 1,2,3 and 4, are not the fixed beacons point. They are the mobile beacons placed close.
# The comparison will be done between 1 marvelmind and 1.2 real (mobile beacon pose on map).
bags_path="/persistent/scripts/bags/correct_better_zone_marvelmind_bag_data_without_distances/"
[X_without_distance,Y_without_distance,Z_without_distance] = get_column_average("/persistent/scripts/bags/correct_better_zone_marvelmind_bag_data_without_distances/")

In [4]:
# points 1,2,3 and 4, are not the fixed beacons point. They are the mobile beacons placed close.
# The comparison will be done between 1 marvelmind and 1.2 real (mobile beacon pose on map).
bags_path="/persistent/scripts/bags/correct_better_zone_marvelmind_bag_data_with_distances/"
[X_with_distance,Y_with_distance,Z_with_distance] = get_column_average(bags_path)

In [15]:
plt.plot(X_without_distance,Y_without_distance, 'bx')
for x,y,p in zip(X_without_distance, Y_without_distance,point_array):                                       
    plt.annotate(f"{p}:({x}, {y})", xy=(x,y), xytext=(x, y)) 

NameError: name 'X_without_distance' is not defined

In [38]:
plt.plot(X_with_distance,Y_with_distance, 'ro')
for x,y,p in zip(X_with_distance, Y_with_distance,point_array):                                       
    plt.annotate(f"{p}:({x}, {y})", xy=(x,y), xytext=(x, y)) 

## Defining strategy for comparing points between real and marvelmind program.

1. Create ortogonal axes from (0,0)
2. Compare real to ortogonal axes, see if there is rotation and displacement. Ideally points 1,2 and 4, should form a close orthogonal axes.
    1. Use calculate rotation between fixed beacons positions 1 and 2 and 3 and 4.
    2. If they rotate in the same direction, correction = (angle1+angle2)/2
    3. If they rotate in different directions, correction = -abs(angle1 - angle2)/2
    4. Use a rotation matrix to rotate the points -correction.
    5. Translate the points to make position 1 approximately the origin (0,0)
3. Compare marvelmind program (passing distance) to ortogonal axes.
    1. Apply the same principals used by real.





In [5]:
def define_2d_rotation_matrix(radians):
    rotation_matrix = np.array([[np.cos(radians),-np.sin(radians)],
                            [np.sin(radians),np.cos(radians)]])
    return rotation_matrix

## Marvelmind program where I passed the distances: Rotating and translating points

In [10]:
## Marvelmind results were I passed the distance between the beacons

fixed_beacons_pose = np.array([[0.004,0.005],[10.246,0.360],[10.097,4.107],[-0.140,3.740]])

radians_between_1_and_2=math.atan2(fixed_beacons_pose[1,1] - fixed_beacons_pose[0,1],fixed_beacons_pose[1,0] - fixed_beacons_pose[0,0])
#radians_between_1_and_4=math.atan2(fixed_beacons_pose[3,1] - fixed_beacons_pose[0,1],fixed_beacons_pose[3,0] - fixed_beacons_pose[0,0])
#radians_correction = -abs((radians_between_1_and_2 - (math.pi/2 - radians_between_1_and_4))/2) # get the medium of the angle error
radians_correction = -radians_between_1_and_2 # you want to turn to the opposite side for the correction.
rotation_matrix = define_2d_rotation_matrix(radians_correction)

x_test = fixed_beacons_pose[:,0]
y_test = fixed_beacons_pose[:,1]



corrected_fixed_beacon_x_with_distance = []
corrected_fixed_beacon_y_with_distance = []
i=0
for x,y,p in zip(x_test, y_test,point_array):
    current_point = np.array([x,y]) 
    [new_x,new_y] = rotation_matrix.dot(current_point)
    if i==0:
        translate_y = new_y
        translate_x = new_x
    new_y = new_y - translate_y
    new_x = new_x - translate_x 
    i=1                                      
    corrected_fixed_beacon_x_with_distance.append(new_x)
    corrected_fixed_beacon_y_with_distance.append(new_y)

#    plt.annotate(f'{p}:({new_x:.4f}, {new_y:.4f})', xy=(new_x,new_y), xytext=(new_x, new_y)) 



corrected_mobile_beacon_x_with_distance = []
corrected_mobile_beacon_y_with_distance = []
plt.plot(corrected_mobile_beacon_x_with_distance,corrected_mobile_beacon_y_with_distance, 'bx')



#print(translate_y)
for x,y,p in zip(X_with_distance, Y_with_distance,point_array):
    current_point = np.array([x,y]) 
    [new_x,new_y] = rotation_matrix.dot(current_point)
    new_y = new_y - translate_y
    new_x = new_x - translate_x                                    
    corrected_mobile_beacon_x_with_distance.append(new_x)
    corrected_mobile_beacon_y_with_distance.append(new_y)
    plt.annotate(f'{p}:({new_x:.4f}, {new_y:.4f})', xy=(new_x,new_y), xytext=(new_x, new_y)) 

plt.plot(corrected_mobile_beacon_x_with_distance,corrected_mobile_beacon_y_with_distance, 'ro')
plt.show()


## Real points plot

In [ ]:
## the cpp code which generates the graph considers the positions 1.2,2.2,3.2, and 4.2 the mobile beacon positions, and 1,2,3, and 4 the fixed beacon position.
# however in the bags, the points 1,2,3 and 4 are the positions of the mobile beacons.
# So to make the graph easier to understand, we are going to remove points 1,2,3 and 4 from the real graph, and 
# change the names of points 1.2,2.2,3.2, and 4.2 to 1,2,3, and 4, respectively.





X_temp = [0, 0, 10.25, 10.25, 10.25, 10.25, 0, 0, 0, 5.096, 4.87, 4.87, 4.87, 4.87, 4.87, 4.87, 10.25, 12.25, 14.25, 27.788, -1.8, -3.6]
Y_temp = [0, 0.042, 0.019, 0.046, 3.748, 3.7435, 3.749, 3.7445, 1.91, 3.152, 1.119, 7.3935, -0.406, 1.909, 3.794, 5.2935, 1.909, 1.909, 1.909, 1.909, 1.909, 1.909]
P_temp = [1, 1.2, 2, 2.2, 3, 3.2, 4, 4.2, 5, 6, 7, 7.1, 7.2, 7.3, 7.4, 7.5, 8, 8.1, 8.2, 8.3, 10.1, 10.2]


points_to_exclude = [1,2,3,4]
for i in points_to_exclude:
    temp_index = P_temp.index(i)
    del P_temp[temp_index]
    del X_temp[temp_index]
    del Y_temp[temp_index]


point_to_change_name = [1.2,2.2,3.2,4.2]
for i in point_to_change_name:
    temp_index = P_temp.index(i)
    P_temp[temp_index] = int(P_temp[temp_index])
  




for x,y,p in zip(X_temp, Y_temp,P_temp):
    plt.annotate(f"{p}:({x}, {y})", xy=(x,y), xytext=(x, y))
    plt.plot(x,y, 'ro') 


plt.grid()




    


# plt.show()

[1, 2, 3, 4, 5, 6, 7, 7.1, 7.2, 7.3, 7.4, 7.5, 8, 8.1, 8.2, 8.3, 10.1, 10.2]
